In [ ]:
!pip install -q MDAnalysis MDTraj torch-geometric

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
output_dir = '/content/drive/MyDrive/PATH/'
os.makedirs(output_dir, exist_ok=True)

print(f"Working directory created at: {output_dir}")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import MDAnalysis as mda
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Load the data_frames list from the file
loaded_data_frames = torch.load(os.path.join(output_dir, 'GAT_Two_Class_data.pt'), weights_only=False)

In [ ]:
# Initialize lists to store train and test frames
train_frames, test_frames = [], []

# Define test frame indices:
# For label 0: take frames 99, 199, 299, ... 999 (10 frames)
test_indices_label0 = list(range(1, 1000, 10))
# For label 1: frames are from index 1000 to 1999, so take frames 1099, 1199, ... 1999 (10 frames)
test_indices_label1 = list(range(1001, 2000, 10))

# Combine test indices from both labels
test_indices = test_indices_label0 + test_indices_label1

test_frames = [loaded_data_frames[i] for i in test_indices]
train_frames = [data for i, data in enumerate(loaded_data_frames) if i not in test_indices]

# Check test set size for each label
test_label_0_count = sum([data.y.item() == 0 for data in test_frames])
test_label_1_count = sum([data.y.item() == 1 for data in test_frames])
print(f"Test set size: Label 0: {test_label_0_count}, Label 1: {test_label_1_count}")

In [ ]:
class GATNet(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GATNet, self).__init__()
        self.conv1 = GATConv(in_channels, 8, heads=8, concat=True)
        self.conv2 = GATConv(8 * 8, 16, heads=8, concat=False)
        self.lin = torch.nn.Linear(16, out_channels)

    def forward(self, x, edge_index, edge_weight=None, batch=None):
        x, attn_weights_1 = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.relu(x)
        x, attn_weights_2 = self.conv2(x, edge_index, return_attention_weights=True)
        x = global_mean_pool(x, torch.zeros(x.size(0), dtype=torch.long))
        x = self.lin(x)
        return F.log_softmax(x, dim=1), attn_weights_1, attn_weights_2

In [ ]:
# Update model for two classes (out_channels=2)
model = GATNet(in_channels=3, out_channels=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

train_loader = DataLoader(train_frames, batch_size=1, shuffle=True)

def train_model(loader, model, optimizer, loss_fn, epochs=100, patience=10):
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for data in loader:
            optimizer.zero_grad()
            output, attn_weights_1, attn_weights_2 = model(data.x, data.edge_index)
            loss = loss_fn(output, data.y.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        print(f'Epoch {epoch}, Loss: {avg_loss}')

        # Early stopping and model saving
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(output_dir, 'GAT_Two_Class_model.pt'))
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

# Train the model
train_model(train_loader, model, optimizer, loss_fn)

In [ ]:
# Evaluate on the test set
y_true, y_pred = [], []

for data in test_frames:
    output, _, _ = model(data.x, data.edge_index)
    y_true.append(data.y.item())
    y_pred.append(output.argmax(dim=1).item())

accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")